# Motivation

Machine learning models are widely used in high energy physics and other scientific domains to classify events and extract meaningful patterns from large datasets.  
For this project, we will be using the Higgs dataset from openml. This simulated dataset is a tabular dataset with a binary target - whether a given process is a higgs process (signal) or any other process (background). 21 Features are kinematic properties, while 7 more features are functions of the kinematic properties - high-level features derived to help discriminate betweeen signal and background.

The goal of this project is to get familiar with some general approaches to machine learning, including cleaning and preprocessing the data, studying the data, as well as preparing, training, and testing a deep neural network on the given data.

In the last exercise of this project, we will attempt to generate adversarial examples on a subset of the samples by only changing a single feature at a time. An adversarial sample is a sample that is intentionally adjusted such that it fools the model to misclassify the given datapoint. These adversarial samples can provide insights into the robustness or fragility of our machine learning models.

# 1. Getting the data

**1a.** First, import `openml` and `pandas`     
**1b.** Download the [Higgs dataset](https://www.openml.org/search?type=data&sort=runs&id=4532&status=active) `openml.datasets.get_dataset(4532)`       
**1c.** Separate the dataset into features (X) and target (y) `X_higgs, y_higgs, _, _ = higgs_dataset.get_data(target=higgs_dataset.default_target_attribute)`      
**1d.** Combine the features and the target into a pandas DataFrame     
**1e.** Print the shape, the head, and the columns of the DataFrame

In [ ]:
import openml
import pandas as pd
import numpy as np 
import random
from matplotlib import pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

In [ ]:
# download Higgs dataset and separate into features (X) and target (y) 
higgs_dataset = openml.datasets.get_dataset(4532)
X_higgs, y_higgs, _, _ = higgs_dataset.get_data(target=higgs_dataset.default_target_attribute)

# combine features and target into one DataFrame
df = X_higgs.copy()
df['target'] = y_higgs

# print shape, head and columns of the DataFrame
print(f'The DataFrame has the shape {np.shape(df)}')
print(f'The columns are: {df.columns}')
df.head()

# 2. Cleaning the data

**2a.** Check for and remove rows with missing values (`NaN`)  
**2b.** Check for and remove rows with infinite values (`inf` or `-inf`)  
**2c.** Check for and remove exact duplicate rows  
**2d.** Print the original and cleaned DataFrames shapes (`df.shape`)

In [ ]:
# check if there are naN values and remove the rows with naN values
print(f'The sum of all naN values in the dateframe is {df.isna().values.sum()}.')

df_clean = df.dropna(axis=0)

# check if there are any inf values
print(f'The sum of all inf values in the dateframe is {np.isinf(df_clean).sum().sum()}.')
# since there are no inf values, no rows need to be removed

# check for and remove exact duplicate rows
print(f'The sum of all dublicate rows in the dataframe is {np.sum(df.duplicated())}.')
df_clean = df_clean.drop_duplicates()
print(f'The sum of all naN values, inf values, and dublicate rows in the cleaned DataFrame are {df_clean.isna().values.sum()}, {np.isinf(df_clean).sum().sum()} and {np.sum(df_clean.duplicated())}.')
print(f'The shape of the original DataFrame is {np.shape(df)}, and the shape of the cleaned DataFrame is {np.shape(df_clean)}.') 

# 3. Data Summary and Statistics

**3a.** Print the names and data types of each column (`df.dtypes`)   
**3b.** Show basic statistics (mean, std, min, max, etc.) for numerical columns (`df.describe()`)  
**3c.** Display the distribution of the target variable (`df['target'].value_counts()`)     
**3d.** Plot the distributions of lepton_pT, lepton_eta, and lepton_phi as histograms for the signal and background events. Each feature should be done in a single plot, e.g. plot signal pT and background pT together.

In [ ]:
# names and data types of each column
print(f'The names and data types of all columns are: \n \n{df_clean.dtypes}')
df_clean.describe()

# Display target distribution
print("Target distribution:")
print(df_clean['target'].value_counts())

# plot distribution of lepton_pT, lepton_eta and lepton_phi for signal and background
df_signal = df_clean[df_clean['target'] == 1]
df_background = df_clean[df_clean['target'] == 0]

fig, ax = plt.subplots(1,3, figsize=(17,5))
ax[0].hist(df_signal['lepton_pT'], bins=50, alpha=0.6, label='signal')
ax[0].hist(df_background['lepton_pT'], bins=50, alpha=0.6, label='background')
ax[0].set_xlabel(r'lepton_pT')

ax[1].hist(df_signal['lepton_eta'], bins=50, alpha=0.6, label='signal')
ax[1].hist(df_background['lepton_eta'], bins=50, alpha=0.6, label='background')
ax[1].set_xlabel(r'lepton_eta')

ax[2].hist(df_signal['lepton_phi'], bins=50, alpha=0.6, label='signal')
ax[2].hist(df_background['lepton_phi'], bins=50, alpha=0.6, label='background')
ax[2].set_xlabel(r'lepton_phi')

i = 0
while i <= 2:
    ax[i].legend()
    ax[i].grid()
    i = i + 1

# 6. Simple MLP Model with Keras

**6a.** Define a simple Multi-Layer Perceptron (MLP) model using [Keras](https://keras.io/)  

**6b.** Compile the model with an appropriate optimizer, loss function, and evaluation metric (`accuracy`) 
```python
model.compile(
    optimizer=OPTIMIZER,
    loss=LOSS,
    metrics=['accuracy']
)
```     
Additionally, plot the training and validation loss over the training epochs

**6c.** Print the model summary  
```python
model.summary()
```

**6d.** Train the model on the training set and validate on the validation set  
```python
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE
)
```

**6e.** Evaluate the trained model on the test set and print the test accuracy  
```python
test_loss, test_acc = model.evaluate(X_test, y_test)
```

In [ ]:
# Re-separate features and target from cleaned DataFrame and ensure binary labels are integers
X = df_clean.drop(columns=['target'])
y = df_clean['target'].astype(int)  

# Standardize features using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Check whether preprocessing worked
print(f'Before scaling:')
print(f'\nMean: {np.mean(X.values, axis=0)}')
print(f'\nStd Dev: {np.std(X.values, axis=0)}')

print(f'\nAfter scaling:')
print(f'\nMean: {X_scaled.mean(axis=0)}')
print(f'\nStd Dev: {X_scaled.std(axis=0)}')

After the preprocessing, the mean value of all features is close to $0$ and the standard deviation is $1$ which is expected, therefore the preprocessing worked.

# 5. Split Dataset

**5a.** Split the dataset (features and target/label separately) into training (70%), test (15%) and validation (15%) datasets.     
**5b.** Print the shapes of the resulting arrays (e.g. `X_Train.shape`)

In [ ]:
# First split into train (70%) and temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(X_scaled, y, test_size=0.30, random_state=42, stratify=y)

# Then split temp into validation (15%) and test (15%)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

# 5b. Print resulting shapes
print("Training set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape)
print("Test set shape:", X_test.shape, y_test.shape)


# 6. Simple MLP Model with Keras

**6a.** Define a simple Multi-Layer Perceptron (MLP) model using [Keras](https://keras.io/)  

**6b.** Compile the model with an appropriate optimizer, loss function, and evaluation metric (`accuracy`) 
```python
model.compile(
    optimizer=OPTIMIZER,
    loss=LOSS,
    metrics=['accuracy']
)
```     
Additionally, plot the training and validation loss over the training epochs

**6c.** Print the model summary  
```python
model.summary()
```

**6d.** Train the model on the training set and validate on the validation set  
```python
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE
)
```

**6e.** Evaluate the trained model on the test set and print the test accuracy  
```python
test_loss, test_acc = model.evaluate(X_test, y_test)
```

In [ ]:
# Define MLP model
model = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid') 
])

# Compile the model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Print model summary
model.summary()

# Train model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=128
)

# Evaluate on the test set
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"\nTest Accuracy: {test_acc:.4f}")

# Plot training and validation loss
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.show()


# 7. Feature Importance     
**7a.** Re-train the MLP while dropping a single feature at a time. Do this for all features and get the resulting accuracy on the test data (also drop the same feature there).    
**7b.** Rank and order the features by their importance, where the most important feature is the one that resulted in the biggest (negative) change of the accuracy when being dropped.

In [ ]:
# Convert back to DataFrame for feature manipulation
X_full_df = pd.DataFrame(X_scaled, columns=X.columns)

baseline_model = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])
baseline_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
baseline_model.fit(X_train, y_train, epochs=10, batch_size=128, verbose=0)
_, baseline_accuracy = baseline_model.evaluate(X_test, y_test, verbose=0)

# Retrain while dropping one feature at a time
import copy

feature_importance = {}

for feature in X.columns:
    # Drop the feature
    X_train_drop = pd.DataFrame(X_train, columns=X.columns).drop(columns=[feature])
    X_val_drop = pd.DataFrame(X_val, columns=X.columns).drop(columns=[feature])
    X_test_drop = pd.DataFrame(X_test, columns=X.columns).drop(columns=[feature])

    # Define and train model
    model_drop = Sequential([
        Dense(16, activation='relu', input_shape=(X_train_drop.shape[1],)),
        Dense(8, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model_drop.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model_drop.fit(X_train_drop, y_train, epochs=10, batch_size=128, verbose=0)
    
    # Evaluate and store drop in accuracy
    _, drop_accuracy = model_drop.evaluate(X_test_drop, y_test, verbose=0)
    feature_importance[feature] = baseline_accuracy - drop_accuracy  # Higher drop = more important

# Rank features by importance
importance_sorted = sorted(feature_importance.items(), key=lambda x: x[1], reverse=True)

print("\nFeature Importance Ranking (most important first):")
for feature, drop in importance_sorted:
    print(f"{feature:25s} | Accuracy drop: {drop:.4f}")


# 8. Brute Force Single Feature Attack on MLP

**8a.** Find all test samples that are correctly classified by the previously trained MLP  
**8b.** Randomly select up to 10 of these correctly classified samples  
**8c.** For each selected sample, iterate over all features  
&nbsp;&nbsp;&nbsp;&nbsp;- For each feature, print its valid range (min/max from training set)  
&nbsp;&nbsp;&nbsp;&nbsp;- Systematically change the feature value within its valid range (e.g., 20 steps)  
&nbsp;&nbsp;&nbsp;&nbsp;- After each change, check if the model misclassifies the sample (`mlp_model.predict(...) != true label`)  
**8d.** If misclassification is achieved by changing a single feature, print the details (Sample, feature name, old/new value, prediction, true label)  
**8e.** At the end, print how many out of the 10 selected samples could be misclassified by changing only one feature

In [ ]:
# convert y_test to numpy array to use indexing 
y_test = np.array(y_test)

# Get correctly classified samples
y_pred_probs = model.predict(X_test)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()
correct_indices = np.where(y_pred == y_test)[0]

# Randomly select up to 10 correctly classified samples
num_samples = min(10, len(correct_indices))
selected_indices = np.random.choice(correct_indices, num_samples, replace=False)

# For each selected sample, try changing each feature
X_train_df = pd.DataFrame(X_train, columns=X.columns)
X_test_df = pd.DataFrame(X_test, columns=X.columns)

min_max = {
    col: (X_train_df[col].min(), X_train_df[col].max())
    for col in X.columns
}

successful_attacks = 0

print("\n Brute Force Single Feature Attack Results:\n")

for idx in selected_indices:
    original_sample = X_test_df.iloc[idx].copy()
    true_label = y_test[idx]
    modified = False

    for feature in X.columns:
        original_value = original_sample[feature]
        min_val, max_val = min_max[feature]
        test_values = np.linspace(min_val, max_val, 20)

        for val in test_values:
            modified_sample = original_sample.copy()
            modified_sample[feature] = val
            pred = model.predict(modified_sample.values.reshape(1, -1))[0][0]
            predicted_label = int(pred > 0.5)

            if predicted_label != true_label:
                successful_attacks += 1
                print(f" Misclassified by modifying 1 feature:")
                print(f"  - Sample index: {idx}")
                print(f"  - Feature: {feature}")
                print(f"  - Original value: {original_value:.4f} , New value: {val:.4f}")
                print(f"  - Prediction: {pred:.4f} , Label: {predicted_label}, True: {true_label}")
                modified = True
                break
        if modified:
            break

# Summary of attack success
print(f"\n {successful_attacks}/{num_samples} samples were misclassified by changing only one feature.")